In [7]:
"""
vcf_parser.py
Parse a ClinVar VCF file and load variants into a local SQLite database.
Handles both plain .vcf and gzipped .vcf.gz inputs.
"""

import sqlite3
import logging
from pathlib import Path

import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger(__name__)


def vcf_to_dataframe(vcf_path: str, max_records: int = 100_000) -> pd.DataFrame:
    """
    Stream-parse a VCF file (plain or gzipped) using cyvcf2.
    Returns a DataFrame with one row per variant.

    Note: ClinVar VCFs may emit "[W::vcf_parse] Contig '1' is not defined in
    the header" warnings. These are harmless — index the file with tabix to
    suppress them:
        tabix -p vcf data/clinvar.vcf.gz
    """
    try:
        from cyvcf2 import VCF
    except ImportError:
        raise ImportError("Install cyvcf2: pip install cyvcf2")

    records = []
    vcf = VCF(vcf_path)

    log.info(f"Parsing VCF: {vcf_path}")
    for i, v in enumerate(vcf):
        if i >= max_records:
            log.info(f"Reached max_records limit ({max_records}). Stopping.")
            break

        # Use variant.INFO.get() directly — more robust than dict(v.INFO),
        # which can fail on multi-value or flag-type INFO fields.
        records.append({
            "chrom":      v.CHROM,
            "pos":        v.POS,        # 1-based, matches VCF spec
            "start":      v.start,      # 0-based, for BED/interval math
            "end":        v.end,
            "variant_id": v.ID or ".",
            "ref":        v.REF,
            "alt":        ",".join(v.ALT) if v.ALT else ".",
            "qual":       v.QUAL,
            "filter":     ";".join(v.FILTER) if v.FILTER else "PASS",
            # ClinVar-specific INFO fields via .get() — safe on missing keys
            "clnsig":     v.INFO.get("CLNSIG") or "Unknown",
            "clndn":      v.INFO.get("CLNDN")  or "Unknown",
            "mc":         v.INFO.get("MC")      or "",
            "af_esp":     _float(v.INFO.get("AF_ESP")),
            "af_exac":    _float(v.INFO.get("AF_EXAC")),
            "af_tgp":     _float(v.INFO.get("AF_TGP")),
        })

    vcf.close()
    df = pd.DataFrame(records)
    log.info(f"Parsed {len(df):,} variants.")
    return df


def _float(val):
    try:
        return float(val)
    except (TypeError, ValueError):
        return None


def load_to_sqlite(df: pd.DataFrame, db_path: str = "data/variants.db") -> None:
    """Write the variant DataFrame to a SQLite database."""
    Path(db_path).parent.mkdir(parents=True, exist_ok=True)
    con = sqlite3.connect(db_path)
    df.to_sql("variants", con, if_exists="replace", index=False)
    con.execute("CREATE INDEX IF NOT EXISTS idx_chrom ON variants(chrom)")
    con.execute("CREATE INDEX IF NOT EXISTS idx_clnsig ON variants(clnsig)")
    con.commit()
    con.close()
    log.info(f"Saved {len(df):,} rows to {db_path}")


if __name__ == "__main__":
    VCF_PATH = "clinvar.vcf.gz"
    df = vcf_to_dataframe(VCF_PATH)
    load_to_sqlite(df)

2026-05-31 20:36:17,724 INFO Parsing VCF: clinvar.vcf.gz
[W::vcf_parse] Contig '1' is not defined in the header. (Quick workaround: index the file with tabix.)
2026-05-31 20:36:18,433 INFO Reached max_records limit (100000). Stopping.
2026-05-31 20:36:18,536 INFO Parsed 100,000 variants.
2026-05-31 20:36:18,820 INFO Saved 100,000 rows to data/variants.db


In [16]:
# QC summary in a notebook cell
from qc_stats import enrich_dataframe, print_summary
df_enriched = enrich_dataframe(df)
print_summary(df_enriched)


  Total variants:       100,000
  Ti/Tv ratio:              nan  (expected ~2.0-2.1 for WGS)

  Variant types:
    SNP            95,175
    deletion        3,063
    insertion       1,482
    MNP               264
    unknown            16

  Clinical significance:
    VUS               55,939
    Benign            30,773
    Pathogenic         7,380
    Other              5,908

  Chi-square (pathogenic enrichment by chrom):
    chi2 = 0.00,  p = 1.00e+00



In [17]:
# Generate all figures
from visualize import run_all
run_all(df)

Saved: outputs/variant_types.png
Saved: outputs/clnsig_distribution.png
Saved: outputs/variants_by_chrom.png
No QUAL scores to plot.

Ti/Tv ratio: nan
